In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objs as go

import utility_functions as uf

In [6]:
domain_id_list = uf.df_topics.domain_id.drop_duplicates().sort_values().to_list()
domain_list = []
for domain in domain_id_list:
    field_id_list = uf.df_topics.query(f"domain_id == {domain}").field_id.drop_duplicates().sort_values().to_list()
    field_list = []
    for field in field_id_list:
        subfield_id_list = uf.df_topics.query(f"field_id == {field}").subfield_id.drop_duplicates().sort_values().to_list()
        subfield_list = [{"id": subfield, "length": 0.01} for subfield in subfield_id_list]
        field_list.append({
            "id": field,
            "length": 0.1,
            "branches": subfield_list,
        })
    domain_list.append({
        "id": domain,
        "length": 1,
        "branches": field_list,
    })

subfields_tree = {
    "id": 0,
    "branches": domain_list,
}

# subfields_tree

# Read metrics_data

In [28]:
path = "data/"
df_country_subfield_norm_world_norm = pd.read_csv(path+"df_country_subfield_norm_world.csv", index_col="country")
df_country_subfield_norm = pd.read_csv(path+"df_country_subfield_norm.csv", index_col="country")

# W1 - distance

In [29]:
def get_dist_w1_tree(tree, mu_dict, nu_dict):
    subtree = tree.get("branches", None)
    if subtree is None:
        leave_id = tree["id"]
        edge_length = tree.get("length", None)
        mu_id = mu_dict.get(str(leave_id), 0)
        nu_id = nu_dict.get(str(leave_id), 0)
        return mu_id, nu_id, abs(mu_id - nu_id) * edge_length
    else:
        mu_id_array = np.full(len(subtree), 0, dtype=float)
        nu_id_array = np.full(len(subtree), 0, dtype=float)
        dist_sum = 0
        edge_length = tree.get("length", None)
        for i, branch in enumerate(subtree):
            mu_id_array[i], nu_id_array[i], dist_branch = get_dist_w1_tree(branch, mu_dict, nu_dict)
            dist_sum += dist_branch
        if edge_length is None:
            return mu_id_array.sum(), nu_id_array.sum(), dist_sum
        else:
            return mu_id_array.sum(), nu_id_array.sum(), dist_sum + abs(mu_id_array.sum() - nu_id_array.sum()) * edge_length


In [30]:
get_dist_w1_tree(subfields_tree,
                 mu_dict = df_country_subfield_norm_world_norm.loc["RU"].dropna().to_dict(),
                 nu_dict = df_country_subfield_norm_world_norm.loc["UA"].dropna().to_dict())

(np.float64(0.9999999999999887),
 np.float64(0.9999999999999893),
 np.float64(0.18419546607979043))

# Get country distances

In [31]:
country_list = df_country_subfield_norm_world_norm.index.to_list()
top_20_countries_list = uf.top_n_countries_by_articles(20)
top_20_idx = [country_list.index(c) for c in top_20_countries_list]

In [32]:
country_dist_w1 = np.full((len(country_list), len(country_list)), np.nan, dtype=float)

for i, country1 in enumerate(country_list):
    for j, country2 in enumerate(country_list):
        if i == j:
            country_dist_w1[i, j] = 0
        if i < j:
            _, _, country_dist_w1[i, j] = _, _, country_dist_w1[j, i] = get_dist_w1_tree(
                subfields_tree,
                mu_dict = df_country_subfield_norm.loc[country1].dropna().to_dict(),
                nu_dict = df_country_subfield_norm.loc[country2].dropna().to_dict())
df_country_dist_w1 = pd.DataFrame(country_dist_w1, index=country_list, columns=country_list)

In [33]:
country_dist_w1_world = np.full((len(country_list), len(country_list)), np.nan, dtype=float)

for i, country1 in enumerate(country_list):
    for j, country2 in enumerate(country_list):
        if i == j:
            country_dist_w1_world[i, j] = 0
        if i < j:
            _, _, country_dist_w1_world[i, j] = _, _, country_dist_w1_world[j, i] = get_dist_w1_tree(
                subfields_tree,
                mu_dict = df_country_subfield_norm_world_norm.loc[country1].dropna().to_dict(),
                nu_dict = df_country_subfield_norm_world_norm.loc[country2].dropna().to_dict())
df_country_dist_w1_world = pd.DataFrame(country_dist_w1_world, index=country_list, columns=country_list)

In [34]:
# df_country_dist_w1.to_csv(path+"df_country_dist_w1.csv")
# df_country_dist_w1_world.to_csv(path+"df_country_dist_w1_world.csv")

In [13]:
df_country_dist_w1

,AD,AE,AF,AG,AL,AM,AO,AR,AS,AT,...,VG,VI,VN,VU,WS,XK,YE,ZA,ZM,ZW
AD,0.000000,0.736512,1.181127,1.750879,0.837047,0.952862,0.608872,0.444300,1.148649,0.751203,...,1.264050,1.152030,0.961525,0.947719,0.439753,0.918373,0.581435,0.604572,1.680326,0.637352
AE,0.736512,0.000000,0.592082,1.188669,0.338906,0.646295,0.360902,0.510514,1.179696,0.226248,...,0.976830,0.704552,0.610493,0.760292,0.813304,0.562728,0.395513,0.426511,1.164172,0.398855
AF,1.181127,0.592082,0.000000,0.746156,0.661236,1.162260,0.838912,0.827595,1.132042,0.734263,...,1.411816,0.459150,1.131878,0.818745,1.006051,1.058105,0.903821,0.691767,0.755457,0.681259
AG,1.750879,1.188669,0.746156,0.000000,1.031327,1.412860,1.343631,1.433960,1.785217,1.127589,...,1.685454,0.722722,1.360448,1.499308,1.632802,1.356467,1.404938,1.306303,0.222561,1.282308
AL,0.837047,0.338906,0.661236,1.031327,0.000000,0.713866,0.386659,0.545628,1.372825,0.179485,...,1.055478,0.408326,0.667682,0.945629,0.837624,0.617267,0.425619,0.591642,0.964446,0.433737
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XK,0.918373,0.562728,1.058105,1.356467,0.617267,0.186805,0.559385,0.801146,1.465771,0.518220,...,0.543102,0.914646,0.153738,1.101710,1.073566,0.000000,0.463869,0.746558,1.342478,0.737962
YE,0.581435,0.395513,0.903821,1.404938,0.425619,0.495552,0.232483,0.437415,1.274993,0.325398,...,0.873730,0.731676,0.518158,0.960775,0.697030,0.463869,0.000000,0.556437,1.298537,0.394135
ZA,0.604572,0.426511,0.691767,1.306303,0.591642,0.846123,0.502859,0.215288,0.836066,0.551188,...,1.180966,0.803257,0.792419,0.451896,0.427821,0.746558,0.556437,0.000000,1.279295,0.252580
ZM,1.680326,1.164172,0.755457,0.222561,0.964446,1.412256,1.277674,1.363456,1.773172,1.053356,...,1.650889,0.652859,1.343207,1.480937,1.547691,1.342478,1.298537,1.279295,0.000000,1.181027


In [14]:
df_country_dist_w1_norm = (
    df_country_dist_w1
    .div(df_country_dist_w1.sum(axis=1), axis=0)
)

In [19]:
# x_labels = list(map(str, df_country_dist_w1.columns))
# y_labels = list(map(str, df_country_dist_w1.index))

sub_list = list(map(str, top_20_countries_list))

# country = "AX"
# top_n = 20
# sub_list = df_country_dist_w1.sort_values(country, ascending=True)[[country]].head(top_n).index.to_list()

x_labels = sub_list
y_labels = sub_list

fig = go.Figure(
    data=go.Heatmap(
        # z=df_country_dist_w1.values,
        z=df_country_dist_w1.loc[sub_list, sub_list].values,
        x=x_labels,
        y=y_labels,
        colorscale="Viridis",
        zmin=0,
        zmax=1.1,
    )
)

# Layout
fig.update_layout(
    title="Distances for top-20 countries",
    width=1000,
    # height=20*len(y_labels),  # taller if more rows
    height=40*len(y_labels),  # taller if more rows
    margin=dict(l=100, r=50, t=50, b=100),
    xaxis=dict(tickangle=45, automargin=True),
    yaxis=dict(autorange="reversed", automargin=True)  # keep top-to-bottom ordering
)

fig.show()

In [20]:
uf.get_country_info("SS")

,name,region,sub-region,code
208,South Sudan,Africa,Sub-Saharan Africa,SS


In [21]:
country = "AX"
top_n = 20
df_plot = df_country_dist_w1.sort_values(country, ascending=True)[[country]].head(top_n)
px.bar(df_plot, x=country, title=uf.get_country_info(country).iloc[0, 0])

In [22]:
top_n = 10
countries = uf.top_n_countries_by_articles(top_n)

# Create figure
fig = go.Figure()

for country in countries:
    df_plot = df_country_dist_w1.sort_values(country, ascending=True).head(top_n)
    fig.add_trace(
        go.Bar(
            x=df_plot[country],           # value
            y=np.arange(top_n),
            hovertext=[code+" "+uf.id2name_country[code] for code in df_plot.index],# category
            name=uf.get_country_info(country).iloc[0, 0],
            orientation='h'
        )
    )

# Layout
fig.update_layout(
    barmode='group',       # 'stack' if you prefer stacked bars
    title=f"Top {top_n} categories for selected countries",
    xaxis_title="Value",
    yaxis_title=None,
    height=800,
    margin=dict(l=150),    # leave space for long y labels
)

# Optional: largest bar at top
fig.update_yaxes(autorange="reversed")

fig.show()

In [23]:
df_country_dist_w1_stats = (
    df_country_dist_w1
    .mean()
    .to_frame("mean")
    .assign(median=df_country_dist_w1.median(),
            std=df_country_dist_w1.std(),
            range=df_country_dist_w1.max() - df_country_dist_w1.min(),
            gini=df_country_dist_w1.apply(uf.gini, axis=1),)
)
df_country_dist_w1_stats

,mean,median,std,range,gini
AD,0.963591,0.880340,0.373053,2.220000,0.205167
AE,0.673360,0.572727,0.388958,1.879058,0.315114
AF,0.904485,0.837596,0.336468,1.969608,0.202394
AG,1.269571,1.288103,0.365873,2.220000,0.157735
AL,0.675392,0.589675,0.369263,1.850479,0.300277
...,...,...,...,...,...
XK,0.803081,0.720191,0.396103,1.944700,0.273385
YE,0.694585,0.579555,0.385276,1.888234,0.297314
ZA,0.755881,0.693291,0.387000,1.829396,0.280611
ZM,1.215192,1.233741,0.362876,2.145544,0.165022


In [25]:
px.histogram(df_country_dist_w1_stats, x="mean", height=800)